In [1]:
import json
import logging
from pathlib import Path

import requests
import pandas as pd

In [2]:
from constants import (
    STARTDUST_URL,
    STARTDUST_ELASTIC_INDEX,
    STARDUST_QUERY_RANGE,
)

In [3]:
def count_countries(time_range=STARDUST_QUERY_RANGE):

    query = {
        "size": 0,
        "query": {
            "range": {
                "@timestamp": {
                    "gte": f"now-{time_range}",
                    "lte": "now"
                }
            }
        },
        "aggs": {
            "src_countries": {
                "terms": {
                    "field": "meta.cric.src.country.keyword",
                    "size": 1000  # there aren't 1000 countries
                }
            },
            "dst_countries": {
                "terms": {
                    "field": "meta.cric.dst.country.keyword",
                    "size": 1000  # there aren't 1000 countries
                }
            }
        }
    }

    # logging.warning(query)
    rsp = requests.post(
        f"{STARTDUST_URL}/{STARTDUST_ELASTIC_INDEX}/_search",
        json=query,
        timeout=60
    )

    rsp.raise_for_status()

    aggs = rsp.json()['aggregations']
    src_countries = pd.DataFrame(aggs['src_countries']['buckets'])
    dst_countries = pd.DataFrame(aggs['dst_countries']['buckets'])

    totals = (
        src_countries.merge(dst_countries, on="key", how="outer", suffixes=("_src", "_dst"))
        .rename(columns={"key": "country"})
        .fillna(0)  # if any are missing, uses float (so convert back to int below)
        .astype({"doc_count_src": int, "doc_count_dst": int})
    )
    totals['total'] =totals['doc_count_src'] + totals['doc_count_dst']

    return totals

count_countries()

,country,doc_count_src,doc_count_dst,total
0,Brazil,1055868,64177,1120045
1,Canada,263804,76102,339906
2,Chile,50,23,73
3,China,32982,14251,47233
4,Czechia,335123,290190,625313
5,Estonia,390593,112698,503291
6,France,3074843,2031429,5106272
7,Germany,9257629,10976435,20234064
8,India,11720,9513,21233
9,Israel,12,0,12
